In [1]:
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn import datasets


In [2]:
df=datasets.load_diabetes(as_frame=True).frame

In [3]:
df

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0


In [4]:
X=df.drop(columns="target")
y=df["target"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

In [6]:
scalar=StandardScaler()
y_train_scaled=scalar.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled=scalar.transform(y_test.values.reshape(-1,1)).ravel()

In [7]:
kernels = ["linear", "poly", "sigmoid", "rbf"]

for kernel in kernels:
    
    model = SVR(kernel=kernel)
    model.fit(X_train, y_train_scaled)
    
    y_train_pred_scaled = model.predict(X_train)
    y_test_pred_scaled = model.predict(X_test)

    print(kernel)
    print("train r2:",r2_score(y_train_scaled,y_train_pred_scaled))
    print("test r2:",r2_score(y_test_scaled,y_test_pred_scaled))      
    print("")

    
#The best model found is linear   

linear
train r2: 0.45191229982475245
test r2: 0.4433761323833776

poly
train r2: 0.579092083431054
test r2: 0.24203771038107802

sigmoid
train r2: -19.721193440731305
test r2: -15.316808189576822

rbf
train r2: 0.6596361676267712
test r2: 0.48844443151651884



### Hyperparameter tuning with GridSearchCV 

In [13]:
param_grid={
    "C":[1,2,5,10,50,100],
    "kernel":["linear", "poly", "sigmoid", "rbf"],
    "epsilon":[0.001,0.1,0.2,0.3,0.4,0.5]    
}




In [14]:
from sklearn.model_selection import GridSearchCV

model=SVR()
grid_search=GridSearchCV(model,param_grid,scoring="r2",cv=5)
grid_search.fit(X_train,y_train_scaled)

GridSearchCV(cv=5, estimator=SVR(),
             param_grid={'C': [1, 2, 5, 10, 50, 100],
                         'epsilon': [0.001, 0.1, 0.2, 0.3, 0.4, 0.5],
                         'kernel': ['linear', 'poly', 'sigmoid', 'rbf']},
             scoring='r2')

In [15]:
print("best_params-",grid_search.best_params_)

best_params- {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


#### Creating Best Model using The Best Parameters

In [17]:
best_model=SVR(C=10,kernel="linear",epsilon=0.1)
best_model.fit(X_train,y_train_scaled)

y_train_pred_scaled = best_model.predict(X_train)
y_test_pred_scaled = best_model.predict(X_test)

print("train r2:",r2_score(y_train_scaled,y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled,y_test_pred_scaled)) 


train r2: 0.5151066486918875
test r2: 0.47444183250401084
